# 이미지 생성 모델 복습

이 노트북은 앞에서 학습한 **VAE → GAN → Diffusion → Stable Diffusion → GPT Image 2** 흐름을 한 번에 복습하기 위한 자료입니다.

목표는 각 모델의 세부 수식을 외우는 것이 아니라, **무엇을 입력받아 어떤 방식으로 이미지를 만드는지**를 구분하는 것입니다.

## 1. 생성 모델 전체 비교

| 모델 | 핵심 아이디어 | 생성 과정 | 강점 | 대표 한계 |
|---|---|---|---|---|
| VAE | 잠재분포 학습 | latent를 샘플링해 Decoder로 복원 | 잠재공간 조작, 보간 | 결과가 흐려질 수 있음 |
| GAN | Generator와 Discriminator의 경쟁 | noise → Generator → image | 빠르고 선명한 생성 | 학습 불안정, mode collapse |
| Diffusion | noise 예측과 반복 제거 | random noise → 반복 denoising → image | 품질, 다양성, 조건 제어 | 추론 비용과 시간이 큼 |

큰 흐름만 기억하면 다음과 같습니다.

```text
VAE       : 잠재공간을 학습한다.
GAN       : Generator가 Discriminator를 속이도록 학습한다.
Diffusion : Noise를 예측하고 반복적으로 제거한다.
```

## 2. VAE 복습

VAE(Variational Autoencoder)는 입력 이미지를 하나의 고정된 latent 값으로 압축하는 대신, **latent의 확률분포**를 학습합니다.

```text
Image x
   ↓
Encoder
   ↓
μ, σ
   ↓  sampling
latent z
   ↓
Decoder
   ↓
Reconstructed / Generated Image
```

대표 목적함수는 재구성 항과 KL Divergence 항으로 구성됩니다.

$$
\mathcal{L}_{VAE} = \mathbb{E}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) || p(z))
$$

- 재구성 항: 입력 정보를 잘 복원하도록 함
- KL Divergence: latent 분포를 기준분포에 가깝게 정렬
- 핵심 장점: latent 공간이 연속적이어서 보간과 속성 조작이 쉬움

In [ ]:
# VAE의 reparameterization 아이디어를 아주 단순화한 예제
import torch

mu = torch.tensor([0.5, -0.2])
log_var = torch.tensor([0.0, -0.4])

std = torch.exp(0.5 * log_var)
epsilon = torch.randn_like(std)
z = mu + std * epsilon

print('mu      :', mu)
print('std     :', std)
print('epsilon :', epsilon)
print('latent z:', z)

## 3. GAN 복습

GAN(Generative Adversarial Network)은 두 신경망을 경쟁시키며 학습합니다.

```text
Random Noise z
      ↓
 Generator
      ↓
 Fake Image ─────┐
                 ↓
Real Image → Discriminator → Real / Fake
```

한 mini-batch에서는 보통 두 번의 최적화가 일어납니다.

1. **Discriminator 학습**: 실제 이미지는 1, 생성 이미지는 0으로 분류
2. **Generator 학습**: 생성 이미지를 Discriminator가 1로 판단하도록 학습

GAN의 loss는 일반적인 분류 문제처럼 계속 작아지는지를 보는 것보다, **Generator와 Discriminator의 균형과 실제 생성 결과를 함께 관찰**해야 합니다.

In [ ]:
# GAN의 핵심 구조를 확인하는 최소 예제
import torch
from torch import nn

NOISE_DIM = 64

generator = nn.Sequential(
    nn.Linear(NOISE_DIM, 128),
    nn.LeakyReLU(0.2),
    nn.Linear(128, 28 * 28),
    nn.Tanh(),
    nn.Unflatten(1, (1, 28, 28)),
)

discriminator = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.LeakyReLU(0.2),
    nn.Linear(128, 1),
)

noise = torch.randn(4, NOISE_DIM)
fake_images = generator(noise)
fake_logits = discriminator(fake_images)

print('noise       :', noise.shape)
print('fake images :', fake_images.shape)
print('D output    :', fake_logits.shape)

In [ ]:
# GAN 학습에서 detach()가 필요한 이유 확인
criterion = nn.BCEWithLogitsLoss()

real_images = torch.randn(4, 1, 28, 28)
fake_images = generator(torch.randn(4, NOISE_DIM))

# 1) Discriminator 학습 시 Generator로 gradient가 전달되지 않도록 detach()
real_logits = discriminator(real_images)
fake_logits_for_d = discriminator(fake_images.detach())

d_real_loss = criterion(real_logits, torch.ones_like(real_logits))
d_fake_loss = criterion(fake_logits_for_d, torch.zeros_like(fake_logits_for_d))
d_loss = (d_real_loss + d_fake_loss) / 2

# 2) Generator 학습 시에는 detach()를 사용하지 않는다.
fake_logits_for_g = discriminator(fake_images)
g_loss = criterion(fake_logits_for_g, torch.ones_like(fake_logits_for_g))

print('D loss:', float(d_loss))
print('G loss:', float(g_loss))

### GAN의 대표 문제: Mode Collapse

서로 다른 noise를 넣어도 Generator가 거의 같은 결과만 반복해서 만드는 현상입니다.

```text
Noise A ─┐
Noise B ─┼→ Generator → 거의 같은 Image
Noise C ─┘
```

따라서 GAN 결과를 평가할 때는 단순히 선명한지만 보지 않고 **샘플 다양성**도 함께 확인해야 합니다.

## 4. Diffusion 복습

Diffusion은 학습 이미지에 noise를 섞고, 모델이 그 noise를 예측하도록 학습합니다.

정방향 확산은 다음 식으로 표현할 수 있습니다.

$$
x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon
$$

- $x_0$: 원본 이미지 또는 latent
- $x_t$: t 시점의 noisy data
- $\epsilon$: 정규분포에서 샘플링한 noise
- $\bar{\alpha}_t$: 원본 신호가 남아 있는 비율

생성 시에는 반대로 **random noise에서 시작하여 모델이 예측한 noise를 반복적으로 제거**합니다.

In [ ]:
# Diffusion 정방향 수식 확인
import torch

torch.manual_seed(7)

x0 = torch.tensor([1.0, -1.0])
epsilon = torch.randn_like(x0)
alpha_bar_t = torch.tensor(0.8)

x_t = (
    torch.sqrt(alpha_bar_t) * x0
    + torch.sqrt(1 - alpha_bar_t) * epsilon
)

print('원본 x0 :', x0)
print('noise ε :', epsilon)
print('noisy xt:', x_t)

## 5. Stable Diffusion = Latent Diffusion

Stable Diffusion은 큰 픽셀 공간에서 직접 denoising하지 않고, VAE가 압축한 **latent space**에서 denoising합니다.

```text
Prompt
  ↓
Tokenizer / Text Encoder
  ↓
Text Embedding ─────────────┐
                            ↓
Random Latent → U-Net → Scheduler
                  ↑         ↓
             Cross Attention
                  ↓
            반복 Denoising
                  ↓
             Final Latent
                  ↓
             VAE Decoder
                  ↓
                Image
```

핵심 구성 요소:

- **Text Encoder**: prompt를 embedding으로 변환
- **Cross Attention**: 이미지 생성 과정이 prompt의 어느 token을 참고할지 결정
- **U-Net**: 현재 latent에서 제거해야 할 noise를 예측
- **Scheduler**: 예측된 noise를 이용해 latent를 다음 단계로 갱신
- **VAE Decoder**: 최종 latent를 RGB 이미지로 복원

### Stable Diffusion 생성 결과를 조절하는 주요 값

| 값 | 의미 |
|---|---|
| `prompt` | 생성할 내용과 스타일 |
| `seed` | 최초 random latent의 난수 시작값 |
| `num_inference_steps` | denoising 반복 횟수 |
| `guidance_scale` | prompt 조건을 얼마나 강하게 반영할지 결정 |

비교 실험을 할 때는 **한 번에 한 변수만 변경**하는 것이 중요합니다. 예를 들어 `guidance_scale`의 영향을 보고 싶다면 model, prompt, seed, steps를 모두 고정합니다.

In [ ]:
# 필요 시 먼저 설치
# %pip install -U diffusers transformers accelerate safetensors torch pillow

# Stable Diffusion 실행 예제
# GPU 환경에서 실행하는 것을 권장합니다.

import torch
from diffusers import StableDiffusionPipeline

MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

# CUDA 환경 예시
# pipe = StableDiffusionPipeline.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
# ).to('cuda')

# generator = torch.Generator(device='cpu').manual_seed(42)
# result = pipe(
#     prompt='A serene landscape of mountains during sunrise, watercolor style',
#     num_inference_steps=30,
#     guidance_scale=7.5,
#     generator=generator,
# )
# result.images[0]

## 6. GPT Image 2

GPT Image 2는 이미지 생성 모델을 직접 내려받아 실행하지 않고 OpenAI의 **관리형 API**를 통해 사용하는 방식입니다.

```text
Prompt
  ↓
client.images.generate()
  ↓
GPT Image 2
  ↓
Base64 Image
  ↓
Decode
  ↓
PNG File
```

로컬 GPU가 필요하지 않은 대신 API key, 네트워크, 사용량과 비용 관리가 필요합니다.

In [ ]:
# 필요 시 설치
# %pip install -q openai python-dotenv pillow

import base64
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI()

prompt = '책을 읽는 작은 로봇의 따뜻한 수채화 일러스트'

# 아래 호출은 실제 API 비용이 발생합니다.
# result = client.images.generate(
#     model='gpt-image-2',
#     prompt=prompt,
# )
#
# image_bytes = base64.b64decode(result.data[0].b64_json)
# output_path = Path('outputs/review_gpt_image.png')
# output_path.parent.mkdir(parents=True, exist_ok=True)
# output_path.write_bytes(image_bytes)
# output_path

## 7. 이미지 생성 Prompt 구조화

짧은 prompt도 이미지를 만들 수 있지만, 요구사항을 검증하고 수정하려면 조건을 구조화하는 것이 유리합니다.

추천 구조:

```text
대상:
행동:
구도:
조명:
배경:
표현 방식:
```

예를 들어 같은 '책을 읽는 로봇'이라도 다음처럼 구체화할 수 있습니다.

In [ ]:
simple_prompt = '책을 읽는 작은 로봇의 따뜻한 수채화 일러스트'

structured_prompt = '''
대상: 작은 둥근 로봇 한 대
행동: 나무 책상에 앉아 펼친 책을 읽는 모습
구도: 로봇과 책이 모두 보이는 정면 중간 거리
조명: 왼쪽 창문에서 들어오는 부드러운 아침 햇빛
배경: 식물과 책장이 있는 조용한 작은 도서관
표현 방식: 따뜻한 파스텔 색감의 수채화 동화책 일러스트
'''.strip()

print('[Simple Prompt]')
print(simple_prompt)
print('\n[Structured Prompt]')
print(structured_prompt)

## 8. 최종 정리

```text
VAE
└─ 잠재분포를 학습하고 latent에서 이미지를 복원

GAN
└─ Generator와 Discriminator의 경쟁

Diffusion
└─ Noise를 예측하고 반복적으로 제거

Stable Diffusion
└─ Diffusion을 latent space에서 수행 + text condition

GPT Image 2
└─ 복잡한 모델 실행을 관리형 API로 제공
```

이미지 생성 기술을 공부할 때는 모델 이름을 외우기보다 다음 세 가지를 계속 질문하면 구조를 이해하기 쉽습니다.

1. **처음 입력은 무엇인가?**
2. **모델은 무엇을 학습하거나 예측하는가?**
3. **이미지가 만들어질 때 어떤 과정을 거치는가?**

## 9. 복습 문제

1. VAE가 일반 Autoencoder와 달리 latent를 확률분포로 표현하는 이유는 무엇인가?
2. GAN에서 Discriminator를 학습할 때 `fake_images.detach()`를 사용하는 이유는 무엇인가?
3. Mode Collapse는 어떤 현상인가?
4. Diffusion 학습에서 U-Net이 주로 예측하는 것은 무엇인가?
5. Stable Diffusion이 pixel space 대신 latent space에서 동작하는 이유는 무엇인가?
6. Scheduler와 U-Net의 역할은 어떻게 다른가?
7. `seed`를 고정하는 이유는 무엇인가?
8. `guidance_scale`이 지나치게 높으면 어떤 문제가 생길 수 있는가?
9. GPT Image API 방식과 Stable Diffusion 로컬 실행 방식의 차이는 무엇인가?
10. 이미지 생성 prompt를 구조화하면 어떤 장점이 있는가?